In [1]:
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments, Trainer
from peft import LoraConfig, get_peft_model, TaskType
from datasets import load_dataset

# **NON INstructional Finetuning**

In [21]:
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments, Trainer
from peft import LoraConfig, get_peft_model, TaskType
from datasets import load_dataset


model = "TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T"


tokenizer = AutoTokenizer.from_pretrained(model)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token


import zipfile
import os
# Path to your zip file
zip_path = "/content/tinyllama-non-instruction.zip"

# Extract all files
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall()


model_path = "/content/checkpoint-5"

# Upgrade torchao to meet PEFT requirements
!pip install --upgrade torchao


non_instruction_model = AutoModelForCausalLM.from_pretrained(model_path, device_map="auto")
# Explicitly move the model to CUDA to resolve the device mismatch warning
non_instruction_model.to('cuda')


prompt = "Clinical trials demonstrated that combining Atorvastatin with Ezetimibe"


inputs = tokenizer(prompt, return_tensors="pt").to("cuda")


outputs = non_instruction_model.generate(
    **inputs,
    max_new_tokens=100,
    temperature=0.8,
    top_p=0.9,
    do_sample=True,
    repetition_penalty=1.1
)


print("\nModel Output:\n")
print(tokenizer.decode(outputs[0], skip_special_tokens=True))

Current model requires 256 bytes of buffer for offloaded layers, which seems does not fit any GPU's remaining memory. If you are experiencing a OOM later, please consider using offload_buffers=True.


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/88 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/transformers/generation/utils.py:2637: UserWarning: You are calling .generate() with the `input_ids` being on a device type different than your model's device. `input_ids` is on cuda, whereas the model is on cpu. You may experience unexpected behaviors or slower generation. Please make sure that you have put `input_ids` to the correct device by calling for example input_ids = input_ids.to('cpu') before running `.generate()`.
  warnings.warn(


RuntimeError: Expected all tensors to be on the same device, but got index is on cuda:0, different from other tensors on cpu (when checking argument in method wrapper_CUDA__index_select)

# **Custom Data**

In [5]:
from datasets import load_dataset

dataset = load_dataset("csv", data_files="/content/pharma_instruction_data.csv",split="train")
dataset


Generating train split: 0 examples [00:00, ? examples/s]

Dataset({
    features: ['instruction', 'input', 'output'],
    num_rows: 5
})

In [6]:
# Function to format dataset examples into instruction-tuning prompts
def format_example(example):

    # Create a structured prompt template
    # Commonly used for supervised fine-tuning (SFT)
    prompt = (
        f"### Instruction:\n"
        f"{example['instruction']}\n"

        # Optional input/context section
        f"### Input:\n"
        f"{example['input']}\n"

        # Expected model response/output
        f"### Response:\n"
        f"{example['output']}"
    )

    # Return formatted text as a dictionary
    # The "text" field is typically used for tokenization
    return {
        "text": prompt
    }

In [7]:
dataset = dataset.map(format_example)
dataset

Map:   0%|          | 0/5 [00:00<?, ? examples/s]

Dataset({
    features: ['instruction', 'input', 'output', 'text'],
    num_rows: 5
})

In [8]:
dataset['text'][0]

'### Instruction:\nExplain the mechanism of action of Metformin.\n### Input:\nNone\n### Response:\nMetformin activates AMP-activated protein kinase (AMPK), which increases glucose uptake and fatty-acid oxidation while inhibiting hepatic gluconeogenesis, thereby lowering blood glucose.'

In [9]:
model = "TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T"

In [10]:

from transformers import AutoTokenizer

# Load the tokenizer for the pretrained model
tokenizer = AutoTokenizer.from_pretrained(model)

# Check whether the tokenizer has a padding token
if tokenizer.pad_token is None:

    # Use the EOS (End Of Sequence) token as the padding token
    # Common practice for decoder-only models like Llama/GPT
    tokenizer.pad_token = tokenizer.eos_token

In [11]:
# Function to tokenize a single dataset example
def tokenize_fn(example):

    # Convert text into token IDs
    tokens = tokenizer(

        # Input text field from the dataset
        example["text"],

        # Truncate sequences longer than max_length
        truncation=True,

        # Pad sequences to a fixed length
        # Useful for batching during training
        padding="max_length",

        # Maximum allowed sequence length
        max_length=512
    )

    # Create labels for causal language modeling
    # The model learns to predict the next token
    # Labels are typically identical to input_ids
    tokens["labels"] = tokens["input_ids"].copy()

    # Return tokenized data
    return tokens

In [12]:
tokenized = dataset.map(tokenize_fn, batched=True)

Map:   0%|          | 0/5 [00:00<?, ? examples/s]

# **LORA**

In [13]:
from peft import LoraConfig, get_peft_model, TaskType

In [14]:
from peft import LoraConfig, TaskType

# Configure LoRA (Low-Rank Adaptation) settings
lora_config = LoraConfig(

    # Specify the type of task
    # CAUSAL_LM is used for autoregressive language models
    # such as GPT, Llama, Mistral, Falcon, etc.
    task_type=TaskType.CAUSAL_LM,

    # Rank of the low-rank matrices
    # Smaller values use less memory
    # Larger values increase learning capacity
    r=8,

    # Scaling factor for LoRA updates
    # Helps control the impact of adapter weights
    lora_alpha=16,

    # Dropout applied to LoRA layers during training
    # Helps reduce overfitting
    lora_dropout=0.05,

    # Layers where LoRA adapters will be inserted
    # q_proj = query projection
    # v_proj = value projection
    target_modules=["q_proj", "v_proj"],

    # Do not train bias parameters
    # Keeps fine-tuning lightweight and efficient
    bias="none"
)

In [15]:
!pip install --upgrade torchao

In [16]:
from transformers import AutoModelForCausalLM

base_model = AutoModelForCausalLM.from_pretrained(model)
instruction_model = get_peft_model(base_model, lora_config)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [17]:
# #STEP A: Load base model
# model = AutoModelForCausalLM.from_pretrained(...)
# #STEP B: Load Stage-1 LoRA
# model = PeftModel.from_pretrained(model, non_instruction_model)
# model = model.merge_and_unload()
# # Why merge?
# # → Model clean + LoRA applied inside weights
# # → Now you can attach SECOND LoRA cleanly
# #STEP C: Attach NEW LoRA for instruction
# instruction_model_lora = get_peft_model(model, lora_config)

In [18]:
from transformers import TrainingArguments

# Define training configuration
args = TrainingArguments(

    # Directory where model checkpoints and logs will be saved
    output_dir="./tinyllama-instruction",

    # Number of full passes through the training dataset
    num_train_epochs=3,

    # Batch size per GPU/device during training
    # Small batch sizes help fit large models into limited VRAM
    per_device_train_batch_size=1,

    # Accumulate gradients over multiple steps before updating weights
    # Effective batch size = 1 × 8 = 8
    gradient_accumulation_steps=8,

    # Learning rate for optimizer
    # Common LoRA fine-tuning learning rate
    learning_rate=2e-4,

    # Enable mixed precision (16-bit floating point)
    # Reduces GPU memory usage and speeds up training
    fp16=True,

    # Print training logs every 20 steps
    logging_steps=20,

    # Keep only the most recent checkpoint
    # Helps save disk space
    save_total_limit=1,

    # Disable logging integrations like WandB or TensorBoard
    report_to="none"
)

In [19]:
from transformers import Trainer

# Create a Trainer object for fine-tuning
trainer = Trainer(

    # LoRA-adapted instruction-tuned model
    # This is the model that will be trained
    model=instruction_model,

    # Training configuration
    # Includes epochs, batch size, learning rate,
    # logging settings, checkpoint saving, etc.
    args=args,

    # Tokenized training dataset
    # Contains input_ids, attention_mask, and labels
    train_dataset=tokenized,
)

# The Trainer API automatically handles:
# - Forward and backward passes
# - Gradient accumulation
# - Optimizer steps
# - Mixed precision training (fp16)
# - Checkpoint saving
# - Logging progress
# - Device placement (GPU/CPU)

In [22]:
trainer.train()

Step,Training Loss


TrainOutput(global_step=3, training_loss=9.961331685384115, metrics={'train_runtime': 5.4278, 'train_samples_per_second': 2.764, 'train_steps_per_second': 0.553, 'total_flos': 47722235166720.0, 'train_loss': 9.961331685384115, 'epoch': 3.0})

In [23]:
model_path = "/content/tinyllama-instruction/checkpoint-3"

In [24]:
from transformers import AutoModelForCausalLM

# Load the fine-tuned instruction model
instruction_model = AutoModelForCausalLM.from_pretrained(

    # Path to the saved model directory
    model_path,

    # Automatically place model layers on available devices
    # (GPU/CPU) based on memory availability
    device_map="auto"
)




Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/88 [00:00<?, ?it/s]

In [25]:
prompt = "Clinical trials demonstrated that combining Atorvastatin with Ezetimibe"


#Tokenize the input prompt
inputs = tokenizer(

    # Convert text prompt into token IDs
    prompt,

    # Return PyTorch tensors

    return_tensors="pt"

# Move tensors to GPU for faster inference
).to("cuda")

In [26]:
# Generate a response from the instruction-tuned model
outputs = instruction_model.generate(

    # Pass tokenized input tensors to the model
    # Includes input_ids and attention_mask
    **inputs,

    # Maximum number of new tokens to generate
    # Stops generation after 100 newly generated tokens
    max_new_tokens=100,

    # Controls randomness in generation
    # Lower values = more focused/deterministic
    # Higher values = more creative/random
    temperature=0.8,

    # Nucleus sampling:
    # Select tokens from the smallest probability set
    # whose cumulative probability is at least 0.9
    top_p=0.9,

    # Enable probabilistic sampling
    # Required for temperature and top_p sampling
    do_sample=True,

    # Penalize repeated tokens/phrases
    # Helps reduce repetitive responses
    repetition_penalty=1.1
)

In [27]:
print("\nModel Output:\n")
print(tokenizer.decode(outputs[0], skip_special_tokens=True))


Model Output:

Clinical trials demonstrated that combining Atorvastatin with Ezetimibe can reduce LDL-C by 35% compared to Ezetimibe alone, but this reduction is offset by a 10% increase in serum triglycerides.
Differences between the formulations: In terms of safety and efficacy, there are no differences between the generic versions of atorvastatin or the branded version of atorvastatin. However, the branded version of atorvastatin has more available information on side


In [32]:
!pip install bitsandbytes accelerate
!pip install --upgrade transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 12.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 50.4 MB/s eta 0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0


In [1]:
!pip install -U bitsandbytes>=0.46.1

In [8]:
# =========================================================
# IMPORT REQUIRED LIBRARIES
# =========================================================

# Hugging Face dataset loader
from datasets import load_dataset

# Transformers components
from transformers import (
    AutoTokenizer,                      # Converts text -> tokens
    AutoModelForCausalLM,              # Loads causal language model
    TrainingArguments,                 # Training configuration
    Trainer,                           # Hugging Face trainer
    BitsAndBytesConfig,                # 4-bit quantization config
    DataCollatorForLanguageModeling    # Dynamic batch padding
)

# PEFT (Parameter Efficient Fine Tuning)
from peft import (
    LoraConfig,        # LoRA configuration
    get_peft_model,   # Apply LoRA adapters
    TaskType,         # Specify task type
    PeftModel         # Load trained LoRA adapters
)

# PyTorch
import torch


# =========================================================
# LOAD CSV DATASET
# =========================================================

# CSV should contain:
# instruction,input,output

dataset = load_dataset(
    "csv",
    data_files="/content/pharma_instruction_data.csv",
    split="train"
)

# Print dataset information
print(dataset)


# =========================================================
# LOAD TOKENIZER
# =========================================================

# TinyLlama instruction/chat model
model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Some tokenizers do not have pad token
# So we use EOS token as padding token
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token


# =========================================================
# FORMAT DATASET USING CHAT TEMPLATE
# =========================================================

# Convert CSV rows into chat format
def format_example(example):

    # Extract fields from dataset
    instruction = example["instruction"]
    input_text = example["input"]
    output_text = example["output"]

    # If input exists
    if input_text and input_text.lower() != "none":

        user_message = f"{instruction}\n\nInput:\n{input_text}"

    else:
        user_message = instruction

    # Chat conversation format
    messages = [
        {
            "role": "user",
            "content": user_message
        },
        {
            "role": "assistant",
            "content": output_text
        }
    ]

    # Apply TinyLlama chat template
    formatted_text = tokenizer.apply_chat_template(
        messages,
        tokenize=False
    )

    return {"text": formatted_text}


# Apply formatting to all rows
dataset = dataset.map(format_example)

# Print one formatted sample
print("\nFormatted Example:\n")
print(dataset[0]["text"])


# =========================================================
# TOKENIZATION + RESPONSE MASKING
# =========================================================

# We only train model on assistant responses
# User prompt tokens are masked using -100

def tokenize_function(example):

    # Get formatted text
    text = example["text"]

    # Convert text -> token ids
    tokenized = tokenizer(
        text,
        truncation=True,
        max_length=512,
        padding="max_length"
    )

    input_ids = tokenized["input_ids"]

    # Assistant tag in chat template
    assistant_text = "<|assistant|>"

    # Find assistant start position
    assistant_start = text.find(assistant_text)

    # If assistant tag not found
    if assistant_start == -1:

        # Ignore all tokens
        labels = [-100] * len(input_ids)

    else:

        # Everything before assistant response
        prefix_text = text[:assistant_start] + assistant_text

        # Count tokens before response starts
        response_token_start = len(
            tokenizer(
                prefix_text,
                add_special_tokens=False
            )["input_ids"]
        )

        # Copy input ids into labels
        labels = input_ids.copy()

        # Mask user/system prompt tokens
        labels[:response_token_start] = [-100] * response_token_start

    # Add labels into tokenized output
    tokenized["labels"] = labels

    return tokenized


# Tokenize complete dataset
tokenized_dataset = dataset.map(
    tokenize_function,

    # Remove original columns
    remove_columns=dataset.column_names
)

print("\nTokenization completed.")


# =========================================================
# 4-BIT QUANTIZATION CONFIG
# =========================================================

# Reduces GPU memory usage significantly

quantization_config = BitsAndBytesConfig(

    # Enable 4-bit loading
    load_in_4bit=True,

    # NF4 quantization type
    bnb_4bit_quant_type="nf4",

    # Computation datatype
    bnb_4bit_compute_dtype=torch.float16,

    # Disable nested quantization
    bnb_4bit_use_double_quant=False,
)


# =========================================================
# LOAD BASE MODEL
# =========================================================

base_model = AutoModelForCausalLM.from_pretrained(

    model_name,

    # Apply quantization
    quantization_config=quantization_config,

    # Automatically place layers on GPU
    device_map="auto"
)

# Disable cache during training
base_model.config.use_cache = False

# Tensor parallelism setting
base_model.config.pretraining_tp = 1


# =========================================================
# LORA CONFIGURATION
# =========================================================

# LoRA trains only small adapter matrices
# instead of full model weights

lora_config = LoraConfig(

    # Task type
    task_type=TaskType.CAUSAL_LM,

    # Rank of LoRA matrices
    r=8,

    # Scaling factor
    lora_alpha=16,

    # Dropout for regularization
    lora_dropout=0.05,

    # No bias training
    bias="none",

    # Transformer layers to modify
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj"
    ]
)


# =========================================================
# APPLY LORA TO MODEL
# =========================================================

model = get_peft_model(
    base_model,
    lora_config
)

# Print trainable parameter count
model.print_trainable_parameters()


# =========================================================
# TRAINING CONFIGURATION
# =========================================================

training_args = TrainingArguments(

    # Save directory
    output_dir="./tinyllama-pharma",

    # Number of epochs
    num_train_epochs=5,

    # Batch size per GPU
    per_device_train_batch_size=1,

    # Simulate larger batch size
    gradient_accumulation_steps=8,

    # Learning rate
    learning_rate=5e-5,

    # Print logs every 10 steps
    logging_steps=10,

    # Save model every epoch
    save_strategy="epoch",

    # Mixed precision training
    fp16=True,

    # Memory efficient optimizer
    optim="paged_adamw_32bit",

    # Warmup percentage
    warmup_ratio=0.05,

    # Cosine learning rate scheduler
    lr_scheduler_type="cosine",

    # Disable wandb/tensorboard
    report_to="none"
)


# =========================================================
# DATA COLLATOR
# =========================================================

# Handles dynamic padding during batching

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)


# =========================================================
# CREATE TRAINER
# =========================================================

trainer = Trainer(

    # LoRA model
    model=model,

    # Training arguments
    args=training_args,

    # Dataset
    train_dataset=tokenized_dataset,

    # Data collator
    data_collator=data_collator
)


# =========================================================
# START TRAINING
# =========================================================

trainer.train()


# =========================================================
# SAVE TRAINED MODEL
# =========================================================

save_path = "/content/tinyllama-pharma"

# Save LoRA adapter weights
trainer.model.save_pretrained(save_path)

# Save tokenizer
tokenizer.save_pretrained(save_path)

print(f"\nModel saved to: {save_path}")


# =========================================================
# LOAD MODEL FOR INFERENCE
# =========================================================

print("\nLoading trained adapter for inference...\n")

# Reload original base model
base_model = AutoModelForCausalLM.from_pretrained(

    model_name,

    device_map="auto",

    torch_dtype=torch.float16
)

# Load trained LoRA adapter
trained_model = PeftModel.from_pretrained(
    base_model,
    save_path
)

# Set model to evaluation mode
trained_model.eval()


# =========================================================
# TEST PROMPT
# =========================================================

messages = [
    {
        "role": "user",
        "content": "What is Ezetimibe?"
    }
]

# Convert messages -> prompt text
prompt = tokenizer.apply_chat_template(
    messages,

    tokenize=False,

    # Adds assistant generation token
    add_generation_prompt=True
)

# Convert prompt -> tensors
inputs = tokenizer(
    prompt,
    return_tensors="pt"
).to("cuda")


# =========================================================
# GENERATE RESPONSE
# =========================================================

with torch.no_grad():

    outputs = trained_model.generate(

        # Input tensors
        **inputs,

        # Maximum generated tokens
        max_new_tokens=100,

        # Creativity/randomness
        temperature=0.7,

        # Nucleus sampling
        top_p=0.9,

        # Enable sampling
        do_sample=True,

        # Reduce repetition
        repetition_penalty=1.1,

        # Padding token
        pad_token_id=tokenizer.eos_token_id
    )


# Decode generated tokens -> text
response = tokenizer.decode(
    outputs[0],
    skip_special_tokens=True
)

# Print response
print("\nMODEL RESPONSE:\n")
print(response)

Dataset({
    features: ['instruction', 'input', 'output'],
    num_rows: 5
})


Map:   0%|          | 0/5 [00:00<?, ? examples/s]


Formatted Example:

<|user|>
Explain the mechanism of action of Metformin.</s>
<|assistant|>
Metformin activates AMP-activated protein kinase (AMPK), which increases glucose uptake and fatty-acid oxidation while inhibiting hepatic gluconeogenesis, thereby lowering blood glucose.</s>



Map:   0%|          | 0/5 [00:00<?, ? examples/s]


Tokenization completed.


...

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!



Model saved to: /content/tinyllama-pharma

Loading trained adapter for inference...



Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

[transformers] Both `max_new_tokens` (=100) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



MODEL RESPONSE:

<|user|>
What is Ezetimibe?
<|assistant|>
Ezetimibe (brand name: Simvastatin) is a medication used to treat cholesterol levels in the blood. It works by inhibiting the activity of certain enzymes that are involved in the production and breakdown of cholesterol. By reducing the amount of LDL (bad) cholesterol in the blood, ezetimibe helps to lower total cholesterol levels and improve lipid profile. In addition, it may also


In [3]:
questions = [
    "Explain the mechanism of action of Metformin.",
    "List two advantages of combining Atorvastatin with Ezetimibe.",
    "Summarize how mRNA vaccines work and mention one current research focus."
]

In [9]:
for q in questions:
    print("Question:", q)
    print("\n--- Non-instruction model ---")
    inputs = tokenizer(q, return_tensors="pt").to("cuda")
    outputs = non_instructional_trained_model.generate(**inputs, max_new_tokens=80)
    print(tokenizer.decode(outputs[0], skip_special_tokens=True))

    print("\n--- Instruction-tuned model ---")
    prompt = f"### Instruction:\n{q}\n### Input:\n\n### Response:\n"
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    outputs = trained_model.generate(**inputs, max_new_tokens=100)
    print(tokenizer.decode(outputs[0], skip_special_tokens=True))
    print("="*80, "\n")

[transformers] Both `max_new_tokens` (=80) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Question: Explain the mechanism of action of Metformin.

--- Non-instruction model ---


[transformers] Both `max_new_tokens` (=100) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Explain the mechanism of action of Metformin.
Metformin is a dipeptide derivative of glucose-6-phosphate dehydrogenase (G6PD) that is a key enzyme in the metabolism of glucose. It is a potent inhibitor of the enzyme, which is responsible for the conversion of glucose to gluconeogenesis.

--- Instruction-tuned model ---


[transformers] Both `max_new_tokens` (=80) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...

List two advantages of combining Atorvastatin with Ezetimibe.
Answer: Atorvastatin is a statin drug that is used to lower cholesterol. Ezetimibe is a drug that is used to lower cholesterol. Combining these two drugs together will help to lower cholesterol.
List two disadvantages of combining Atorvastatin with Ezetimibe.
Answer: At

--- Instruction-tuned model ---


[transformers] Both `max_new_tokens` (=80) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


### Instruction:
List two advantages of combining Atorvastatin with Ezetimibe.
### Input:

### Response:
Advantages of combining Atorvastatin with Ezetimibe:
1. Reduced risk of heart attack and stroke
2. Improved cholesterol levels
3. Reduced risk of developing heart disease
4. Lowered risk of developing atherosclerosis
5. Reduced risk of developing gallstones
6. Reduced risk of developing pancreatitis
7. Reduced risk of developing gout

Question: Summarize how mRNA vaccines work and mention one current research focus.

--- Non-instruction model ---


[transformers] Both `max_new_tokens` (=100) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Summarize how mRNA vaccines work and mention one current research focus.
What is the difference between mRNA and DNA?
DNA is a long, thread-like molecule that carries genetic information. Messenger RNA (mRNA) is a short, thread-like molecule that carries the instructions for making proteins.
What is the difference between mRNA and DNA?
DNA is a long

--- Instruction-tuned model ---
### Instruction:
Summarize how mRNA vaccines work and mention one current research focus.
### Input:

### Response:
Messenger RNA (mRNA) vaccines work by using a piece of genetic material called mRNA to instruct the body's cells to produce a protein that mimics the spike protein on the surface of the SARS-CoV-2 virus. This protein is what the virus uses to enter and infect cells, and it is the target of the vaccine. The mRNA vaccine is injected into the body, and

